# Perfiles de participación ciudadana

**Equipo Utrinos** · Reto 1 del hackatón del INE "Hackea la participación"

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/carrapido/perfiles-participacion-ine/blob/main/notebooks/perfiles_participacion.ipynb)

Este notebook reproduce el análisis completo con los Conteos Censales de Participación Ciudadana (CCPC) del INE: la calidad de los datos, el panorama de la brecha entre mujeres y hombres, la segmentación en cinco perfiles y su validación con la elección judicial de 2025.

- **Para correrlo en Colab:** *Entorno de ejecución → Ejecutar todo*. No hay que preparar nada: la primera celda clona el repositorio público, que trae los datos agregados.
- **No copia el cálculo:** lo importa de `src/segmentacion.py` (el modelo) y de `scripts/exportar_web.py` (los datos del sitio). Así sus cifras son, por construcción, las mismas del sitio y del documento técnico.
- **Se comprueba solo:** las celdas con `assert` se detienen si alguna cifra no coincide con los resultados guardados o con el documento técnico.

Sitio: https://carrapido.github.io/perfiles-participacion-ine/

## 1. Configuración

En Colab, la celda clona el repositorio en `/content` (unos 65 MB, con los datos agregados). En una copia local usa la carpeta del repositorio. Colab ya trae pandas, NumPy, Plotly e ipywidgets, así que no instala nada.

In [ ]:
import json
import math
import os
import subprocess
import sys
import tempfile
from pathlib import Path

REPOSITORIO = "https://github.com/carrapido/perfiles-participacion-ine.git"
EN_COLAB = "google.colab" in sys.modules


def raiz_del_repositorio() -> Path:
    """En Colab clona el repositorio la primera vez; en local, el notebook vive en notebooks/."""
    if EN_COLAB:
        raiz = Path("/content/perfiles-participacion-ine")
        if not raiz.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPOSITORIO, str(raiz)], check=True)
        return raiz
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


RAIZ = raiz_del_repositorio()
os.chdir(RAIZ)  # los scripts leen data/ con rutas relativas a la raíz del repositorio
sys.path[:0] = [str(RAIZ / "src"), str(RAIZ / "scripts")]
print("Colab" if EN_COLAB else "Local", "→", RAIZ)

In [ ]:
import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display

import exportar_web
from segmentacion import (ANIOS, HOMBRES, MUJERES, VARIABLES, cargar_celdas, cargar_datos, describir_perfiles,
                          metricas_por_k, tasa_de_participacion)

AGREGADOS = "data/agregados"
RESULTADOS = "data/resultados/por_sexo"
JUDICIAL = 2025

# Los mismos colores del sitio: morado para las mujeres, café para los hombres, verde para la judicial
COLOR_MUJERES, COLOR_HOMBRES = "#5e3c99", "#b35806"
COLOR_JUDICIAL, COLOR_CONTEXTO = "#008300", "#8a8984"
pio.templates.default = "simple_white"


def nueva_figura(titulo: str, eje_y: str) -> go.Figure:
    figura = go.Figure()
    figura.update_layout(title=titulo, yaxis_title=eje_y, height=440, hovermode="x unified",
                         legend={"orientation": "h", "y": -0.15})
    figura.update_yaxes(showgrid=True)
    return figura

## 2. Carga y calidad de los datos

`scripts/agregar_ccpc.py` sumó los CSV crudos de cada elección (una fila por sección × sexo × edad) en `data/agregados/`, y guardó un control de calidad por elección.

- **Participación:** SV / (SV + NV), en porcentaje. SV son las personas con marca de voto y NV las que no la tienen. Las de registro no especificado (NS) quedan fuera porque su proporción cambia mucho entre años.
- **Brecha:** participación de las mujeres menos la de los hombres, en puntos. El sexo no binario se excluye de la brecha.
- La judicial de 2025 no trae Campeche: el archivo oficial del INE no lo incluye.

In [ ]:
ELECCIONES = [*map(str, ANIOS), "PJ2025"]  # como se llaman los archivos de data/agregados/


def calidad_de(eleccion: str) -> dict:
    with open(f"{AGREGADOS}/calidad_{eleccion}.json", encoding="utf-8") as archivo:
        return json.load(archivo)


calidad = pd.DataFrame([calidad_de(eleccion) for eleccion in ELECCIONES]).set_index("anio")
display(calidad[["entidades", "entidades_esperadas", "completo", "LN", "NS_pct", "DEF_vacio_LN_pct",
                 "registros_no_binario_LN"]])

assert calidad.completo.all(), "Falta alguna entidad en los agregados"
assert (calidad.SV + calidad.NV + calidad.NS == calidad.LN).all(), "SV + NV + NS debe sumar la lista nominal"

In [ ]:
conteos_nacionales = exportar_web.cargar_conteos(["AELEC", "SEXO"])
participacion = exportar_web.participacion_por_sexo(conteos_nacionales, ["AELEC"])
sumas = conteos_nacionales.groupby("AELEC")[["SV", "NV"]].sum()
participacion["total"] = tasa_de_participacion(sumas.SV, sumas.NV)
display(participacion.round(2))

# Coinciden con el control de calidad que guardó scripts/agregar_ccpc.py
assert np.allclose(participacion.mujeres, calidad.participacion_mujeres, atol=0.005)
assert np.allclose(participacion.hombres, calidad.participacion_hombres, atol=0.005)

### Comparación con los cómputos oficiales

Participación oficial = votos entre lista nominal de los cómputos del INE. En las federales se usa la elección de diputaciones, salvo en 2012, que usa la presidencial; en la judicial, la de la Suprema Corte. Los cómputos incluyen casillas especiales y voto desde el extranjero, que los CCPC no traen, así que la cifra oficial debe quedar un poco arriba.

Fuentes: [2009](https://web.archive.org/web/20130803003348/http://www.ife.org.mx/documentos/RESELEC/SICEEF/datosXML/02090400.xml) · [2012](https://web.archive.org/web/20120830144813/http://computos2012.ife.org.mx/reportes/presidente/distritalPresidenteEF.html) · [2015](http://computos2015.ine.mx/JSON/DIPUTADO/NACIONAL/diputados_nacional.json) · [2018](https://computos2018.ine.mx/assets/JSON/DIPUTADOS_MR/NACIONAL/Diputados_MR_NACIONAL.json) · [2021](https://computos2021.ine.mx/assets/JSON/DIPUTADOS_MR/NACIONAL/Diputados_MR_NACIONAL.json) · [2024](https://computos2024.ine.mx/assets/JSON/DIPUTADOS_MR/NACIONAL/Diputados_MR_NACIONAL.json) · [judicial 2025](https://computospj2025.ine.mx/assets/data/supremaCorte/estadisticaNacional.json)

In [ ]:
OFICIAL = pd.Series({2009: 44.61, 2012: 63.08, 2015: 47.73, 2018: 63.21, 2021: 52.66, 2024: 60.55, 2025: 13.02})
comparacion = pd.DataFrame({"oficial": OFICIAL, "CCPC": participacion.total})
comparacion["diferencia"] = comparacion.oficial - comparacion.CCPC
display(comparacion.round(2))

assert comparacion.diferencia.between(0, 1).all(), "Los CCPC deben quedar entre 0 y 1 punto abajo de lo oficial"

## 3. Panorama de la brecha

### Por elección

In [ ]:
federales = participacion.loc[ANIOS]
figura = nueva_figura("Participación por sexo en las elecciones federales", "Participación (%)")
for sexo, color in (("mujeres", COLOR_MUJERES), ("hombres", COLOR_HOMBRES)):
    figura.add_scatter(x=ANIOS, y=federales[sexo], name=sexo.capitalize(), mode="lines+markers",
                       line={"color": color, "width": 2}, marker={"size": 8})
figura.update_xaxes(tickvals=ANIOS)
figura.show()

### Por grupo de edad

Cada celda es la brecha de un grupo de edad en una elección. Morado: las mujeres votan más; café: los hombres votan más. La escala se satura en ±20 puntos, como el mapa del sitio. La tabla de abajo da, para cada elección, el primer grupo de edad en que los hombres votan más.

In [ ]:
conteos_por_edad = exportar_web.cargar_conteos(["AELEC", "GEDAD", "SEXO"])
brecha_por_edad = (exportar_web.participacion_por_sexo(conteos_por_edad, ["AELEC", "GEDAD"]).brecha
                   .unstack("AELEC").reindex(exportar_web.GRUPOS_DE_EDAD))
NOMBRE_DE_ELECCION = {**{anio: str(anio) for anio in ANIOS}, JUDICIAL: f"Judicial {JUDICIAL}"}
brecha_por_edad = brecha_por_edad.rename(columns=NOMBRE_DE_ELECCION)

figura = go.Figure(go.Heatmap(z=brecha_por_edad.to_numpy(), x=brecha_por_edad.columns, y=brecha_por_edad.index,
                              colorscale="PuOr", zmin=-20, zmax=20, texttemplate="%{z:.1f}",
                              colorbar={"title": {"text": "puntos"}}))
figura.update_layout(title="Brecha mujeres − hombres por grupo de edad, en puntos", height=560,
                     yaxis={"autorange": "reversed", "title": "Edad"})
figura.show()


def primer_grupo_donde_votan_mas_los_hombres(brecha: pd.Series) -> str:
    negativos = brecha[brecha < 0]
    return negativos.index[0] if len(negativos) else "ninguno"


display(brecha_por_edad.apply(primer_grupo_donde_votan_mas_los_hombres).rename("Desde la edad").to_frame())

### Por tipo de sección

Brecha en puntos, por tipo de sección electoral (urbana, mixta o rural).

In [ ]:
conteos_por_seccion = exportar_web.cargar_conteos(["AELEC", "TIPOSEC", "SEXO"])
brecha_por_seccion = (exportar_web.participacion_por_sexo(conteos_por_seccion, ["AELEC", "TIPOSEC"]).brecha
                      .unstack("TIPOSEC")[["U", "M", "R"]]
                      .rename(columns={"U": "Urbana", "M": "Mixta", "R": "Rural"}, index=NOMBRE_DE_ELECCION))
display(brecha_por_seccion.round(2))

## 4. Segmentación por sexo

- **Unidad de análisis:** la celda municipio × sexo × etapa de vida (18-24, 25-34, 35-44, 45-59, 60-69 y 70 o más). Solo entran las celdas comparables: con datos de los dos sexos en las seis federales y al menos 100 personas en su lista nominal cada año.
- **Variables**, de la participación federal 2009–2024 de cada celda:
  - `nivel`: participación media;
  - `tendencia`: cambio por elección;
  - `ciclo`: cuánto sube en las elecciones presidenciales frente a las intermedias;
  - `ventaja`: diferencia frente al otro sexo de su mismo municipio y edad;
  - `rural`: % de su lista nominal en secciones rurales.
- **Modelo:** k-means ponderado por la lista nominal de 2024, por separado para mujeres y para hombres, para que cada perfil tenga un solo sexo.
- **Cómo se eligió k:** para cada sexo se probaron k de 2 a 5 y se midió la silueta (qué tan separados quedan los grupos) y la estabilidad (ARI medio entre el ajuste completo y ajustes con el 80 % de las celdas).

In [ ]:
datos = cargar_datos(AGREGADOS)
print(f"Celdas comparables: {len(datos.variables):,}")

metricas = pd.DataFrame(metricas_por_k(datos))
display(metricas)

# Las mismas métricas que guardó scripts/perfiles_por_sexo.py
pd.testing.assert_frame_equal(metricas, pd.read_csv(f"{RESULTADOS}/metricas_k.csv"), check_dtype=False)

In [ ]:
for medida, titulo in (("silueta", "Silueta por k (más alta: grupos más separados)"),
                       ("estabilidad_ARI", "Estabilidad por k (ARI; 1 = el mismo agrupamiento)")):
    figura = nueva_figura(titulo, medida)
    for sexo, color in (("mujeres", COLOR_MUJERES), ("hombres", COLOR_HOMBRES)):
        del_sexo = metricas[metricas.sexo == sexo]
        figura.add_scatter(x=del_sexo.k, y=del_sexo[medida], name=sexo.capitalize(), mode="lines+markers",
                           line={"color": color, "width": 2}, marker={"size": 8})
    figura.update_xaxes(tickvals=[2, 3, 4, 5], title="k")
    figura.show()

In [ ]:
K_MUJERES, K_HOMBRES = 2, 3
mas_estable = metricas.loc[metricas.groupby("sexo").estabilidad_ARI.idxmax()].set_index("sexo")
mas_separado = metricas.loc[metricas.groupby("sexo").silueta.idxmax()].set_index("sexo")
display(mas_estable[["k", "silueta", "estabilidad_ARI"]])

# Las dos medidas eligen el mismo k para cada sexo: 2 perfiles de mujeres y 3 de hombres
assert mas_estable.k.to_dict() == mas_separado.k.to_dict() == {"hombres": K_HOMBRES, "mujeres": K_MUJERES}
assert mas_estable.estabilidad_ARI.round(2).to_dict() == {"hombres": 0.97, "mujeres": 0.96}  # documento técnico

## 5. Los cinco perfiles

La elección judicial de 2025 no entra al agrupamiento: aquí solo se mide cuánto votó cada perfil. Los nombres, colores y textos de los perfiles los escribió el equipo en `data/resultados/por_sexo/textos_perfiles.json`.

In [ ]:
judicial = cargar_celdas(f"{AGREGADOS}/agg_PJ2025.csv")
resumen, asignacion = describir_perfiles(datos, K_MUJERES, K_HOMBRES, judicial)

# El mismo resultado que guardó scripts/perfiles_por_sexo.py y que usa el sitio
sufijo = f"k{K_MUJERES}_{K_HOMBRES}"
pd.testing.assert_frame_equal(resumen, pd.read_csv(f"{RESULTADOS}/perfiles_resumen_{sufijo}.csv"), check_dtype=False)
pd.testing.assert_frame_equal(asignacion, pd.read_csv(f"{RESULTADOS}/celdas_{sufijo}.csv"), check_dtype=False)

with open(f"{RESULTADOS}/textos_perfiles.json", encoding="utf-8") as archivo:
    textos = pd.DataFrame(json.load(archivo)["perfiles"]).set_index("grupo_csv").sort_values("orden")
resumen.index = resumen.sexo + "_" + resumen.grupo.astype(str)
perfiles = textos[["nombre", "color"]].join(resumen).set_index("nombre")

COLUMNAS = {"pct_LN_total": "% de la lista nominal", "pct_de_su_sexo": "% de su sexo", "p2024": "Votó en 2024",
            "pPJ2025": "Votó en la judicial 2025", **{variable: variable for variable in VARIABLES}}
display(perfiles[list(COLUMNAS)].rename(columns=COLUMNAS).round(1))

In [ ]:
# Las cifras de los perfiles en el documento técnico, redondeadas a un decimal
DOCUMENTO_TECNICO = {
    "Hombres en rezago persistente": (14.4, 48.2, 8.8),
    "Hombres de voto presidencial": (22.3, 50.3, 9.3),
    "Hombres mayores comprometidos": (11.4, 71.7, 22.0),
    "Mujeres urbanas en ascenso": (28.5, 63.2, 13.4),
    "Mujeres de voto constante": (23.5, 65.4, 14.0),
}
calculadas = {nombre: tuple(fila.round(1)) for nombre, fila in perfiles[["pct_LN_total", "p2024", "pPJ2025"]].iterrows()}
assert calculadas == DOCUMENTO_TECNICO, calculadas

In [ ]:
figura = nueva_figura("Participación de cada perfil en las elecciones federales", "Participación (%)")
for nombre, perfil in perfiles.iterrows():
    figura.add_scatter(x=ANIOS, y=[perfil[f"p{anio}"] for anio in ANIOS], name=nombre, mode="lines+markers",
                       line={"color": perfil.color, "width": 2}, marker={"size": 8})
figura.update_xaxes(tickvals=ANIOS)
figura.show()

## 6. Validación con la elección judicial de 2025

Si los perfiles describen hábitos de participación y no solo una elección, su orden debería sostenerse en una elección que no se usó para construirlos.

In [ ]:
validacion = perfiles[["p2024", "pPJ2025"]].rename(columns={"p2024": "Federal 2024", "pPJ2025": "Judicial 2025"})
validacion["Lugar en 2024"] = validacion["Federal 2024"].rank(ascending=False).astype(int)
validacion["Lugar en la judicial"] = validacion["Judicial 2025"].rank(ascending=False).astype(int)
display(validacion.sort_values("Lugar en 2024").round(1))

assert (validacion["Lugar en 2024"] == validacion["Lugar en la judicial"]).all(), "La judicial cambió el orden"

figura = nueva_figura("Participación de cada perfil: federal 2024 y judicial 2025", "Participación (%)")
for nombre, perfil in perfiles.iterrows():
    figura.add_scatter(x=["Federal 2024", "Judicial 2025"], y=[perfil.p2024, perfil.pPJ2025], name=nombre,
                       mode="lines+markers", line={"color": perfil.color, "width": 2}, marker={"size": 8})
figura.show()

## 7. Exploración por entidad

Elige una entidad y un sexo: la tabla da su participación por elección y la gráfica, qué parte de la lista nominal de ese sexo pertenece a cada perfil.

In [ ]:
participacion_por_entidad = exportar_web.participacion_por_sexo(
    exportar_web.cargar_conteos(["AELEC", "EDOCVE", "SEXO"]), ["EDOCVE", "AELEC"])
reparto_por_entidad = exportar_web.reparto_por(exportar_web.celdas_del_modelo(), ["EDOCVE"])


def explorar(entidad: int, sexo: int) -> None:
    display(Markdown(f"### {exportar_web.ENTIDADES[entidad]}"))
    display(participacion_por_entidad.loc[entidad].rename(index=NOMBRE_DE_ELECCION).round(1))
    reparto = reparto_por_entidad.loc[(entidad, sexo)].reindex(textos.index).dropna()
    figura = go.Figure(go.Bar(x=reparto.to_numpy(), y=textos.nombre[reparto.index], orientation="h",
                              marker_color=textos.color[reparto.index], texttemplate="%{x:.1f}%"))
    figura.update_layout(title="% de su lista nominal 2024 en cada perfil", height=280,
                         xaxis={"range": [0, 100]}, yaxis={"autorange": "reversed"})
    figura.show()


widgets.interact(
    explorar,
    entidad=widgets.Dropdown(options=[(nombre, clave) for clave, nombre in exportar_web.ENTIDADES.items()],
                             description="Entidad"),
    sexo=widgets.ToggleButtons(options=[("Mujeres", MUJERES), ("Hombres", HOMBRES)], description="Sexo"),
);

## 8. Exportación de los datos del sitio

`scripts/exportar_web.py` escribe los JSON que dibuja el sitio. Aquí los genera de nuevo en una carpeta temporal y los compara, valor por valor, con los publicados en `web/public/data/`.

Los textos, las claves y los enteros deben ser idénticos. Los decimales se comparan con una tolerancia relativa de 10⁻¹²: pandas 2.2, el de Colab, no lee de forma exacta algunos números de los CSV y puede cambiar el último de sus 16 dígitos; pandas 3 sí los lee exactos. El sitio los muestra con uno o dos decimales.

In [ ]:
def mismos_datos(a, b) -> bool:
    """Igualdad de dos JSON; dos decimales son iguales si difieren menos que su precisión de lectura."""
    if isinstance(a, dict):
        return a.keys() == b.keys() and all(mismos_datos(a[clave], b[clave]) for clave in a)
    if isinstance(a, list):
        return len(a) == len(b) and all(mismos_datos(x, y) for x, y in zip(a, b))
    if isinstance(a, float) and isinstance(b, float):
        return math.isclose(a, b, rel_tol=1e-12, abs_tol=1e-12)
    return a == b


def leer_json(ruta: Path):
    return json.loads(ruta.read_text(encoding="utf-8"))


with tempfile.TemporaryDirectory() as carpeta:
    exportar_web.exportar(Path(carpeta))
    generados = {archivo.name: leer_json(archivo) for archivo in sorted(Path(carpeta).glob("*.json"))}

distintos = [nombre for nombre, datos in generados.items()
             if not mismos_datos(datos, leer_json(RAIZ / "web/public/data" / nombre))]
print(f"{len(generados)} archivos generados; distintos de los del sitio: {distintos or 'ninguno'}")
assert not distintos, distintos

## Resultado

Si todas las celdas corrieron sin error, el notebook reprodujo los agregados, las métricas de k, los cinco perfiles, la validación con la judicial y los datos del sitio, con las mismas cifras del documento técnico.